#### **Seq2Seq con atención clásica**

Este cuaderno adapta dos líneas modernas de estudio importantes

1. **Self-attention desde cero**: embeddings, matrices $Q$, $K$, $V$, producto punto escalado, máscara causal, multi-head attention y cross-attention.
2. **Variantes modernas de atención en LLMs**: grouped-query attention (GQA), KV cache, sliding-window attention (SWA), atención dispersa tipo top-$k$, una aproximación didáctica a multi-head latent attention (MLA), QK-Norm, partial RoPE, gated attention e idea de atención híbrida.

El objetivo no es reemplazar el enfoque Seq2Seq original, sino mostrar la transición conceptual:

$$
\text{Seq2Seq + atención encoder-decoder} \rightarrow \text{Transformer} \rightarrow \text{LLMs eficientes de contexto largo}
$$

**Referencias base**


- Lilian Weng ,Attention? Attention!, 2018.  
  https://lilianweng.github.io/posts/2018-06-24-attention/ 
- Sebastian Raschka, *Understanding and Coding the Self-Attention Mechanism of Large Language Models From Scratch*, 2023.  
  https://sebastianraschka.com/blog/2023/self-attention-from-scratch.html

### **Modelos seq2seq + atención**


Los modelos de atención han transformado la forma en que abordamos las tareas de traducción automática y otras aplicaciones de procesamiento de lenguaje natural (NLP). Estos modelos permiten que el decodificador se centre en diferentes partes de la secuencia de entrada mientras genera cada palabra de la secuencia de salida.



Dos mecanismos importantes en este contexto son la atención global y la atención local. Este cuaderno detalla estos mecanismos, sus implementaciones, y sus aplicaciones prácticas.

#### **Mecanismo de atención global**

<img src="https://lilianweng.github.io/posts/2018-06-24-attention/encoder-decoder-attention.png" width="600">


**Descripción general**

El mecanismo de atención global, introducido por Bahdanau también conocido como atención suave, permite que el decodificador considere todas las posiciones de la secuencia de entrada al generar cada palabra de la secuencia de salida. Este enfoque asegura que el modelo tenga acceso a toda la información de la entrada en cada paso del proceso de decodificación, mejorando la calidad de la traducción, especialmente en secuencias largas y complejas.

**Ecuaciones y cálculos**

El mecanismo de atención global se basa en los siguientes pasos y ecuaciones:

1. **Cálculo de los puntajes de atención**:
   Para cada paso de tiempo del decodificador, se calcula un puntaje de atención $ e_{ij}$ que mide la afinidad entre el estado oculto del decodificador en el paso de tiempo $ j$, denotado como $ s_{j-1}$, y el estado oculto del codificador en el paso de tiempo $ i$, denotado como $ h_i$. Esto se puede calcular usando una red neuronal feedforward con una sola capa oculta (o cualquier otra función de afinidad):

   $$
   e_{ij} = v^T \tanh(W_1 h_i + W_2 s_{j-1})
   $$

   donde $W_1$ y $W_2$ son matrices de peso aprendibles y $v$ es un vector de peso aprendible.

2. **Normalización de puntajes de atención**:
   Los puntajes de atención se normalizan usando la función softmax para obtener los pesos de atención $ \alpha_{ij}$, que son distribuciones de probabilidad sobre las posiciones de la secuencia de entrada:

 $$
   \alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})}
  $$

3. **Cálculo del vector de contexto**:
   El vector de contexto $ c_j$ para cada paso de tiempo del decodificador se calcula como una combinación ponderada de los estados ocultos del codificador:

 $$
   c_j = \sum_{i=1}^{T_x} \alpha_{ij} h_i
 $$

4. **Generación de la salida del decodificador**:
   Finalmente, el vector de contexto $ c_j$ se combina con el estado oculto del decodificador $ s_j$ para generar la salida $ y_j$:

 $$
   y_j = g(c_j, s_j)
 $$

  donde $ g$ puede ser una función no lineal como una red neuronal.
    


La siguiente implementación en PyTorch muestra cómo se puede construir un mecanismo de atención global dentro de un modelo seq2seq:

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo usado: {dispositivo}")

def crear_mascara_por_longitudes(longitudes, longitud_maxima=None):
    """
    Crea una máscara booleana a partir de las longitudes reales de cada secuencia.

    longitudes: tensor de forma [batch]
    salida: tensor booleano de forma [batch, longitud_maxima]
    """
    if longitud_maxima is None:
        longitud_maxima = int(longitudes.max().item())
    posiciones = torch.arange(longitud_maxima, device=longitudes.device).unsqueeze(0)
    return posiciones < longitudes.unsqueeze(1)

def masked_softmax(puntajes, mascara=None, dim=-1, eps=1e-12):
    """
    Softmax con máscara.

    Esta función evita asignar probabilidad a posiciones de padding o posiciones
    no permitidas por una ventana local.
    """
    if mascara is None:
        return F.softmax(puntajes, dim=dim)

    mascara = mascara.to(dtype=torch.bool, device=puntajes.device)
    puntajes_enmascarados = puntajes.masked_fill(~mascara, -1e9)
    pesos = F.softmax(puntajes_enmascarados, dim=dim)

    # Re-normalización defensiva para evitar residuos numéricos en posiciones inválidas.
    pesos = pesos * mascara.to(dtype=pesos.dtype)
    normalizador = pesos.sum(dim=dim, keepdim=True).clamp_min(eps)
    return pesos / normalizador

class AtencionBahdanau(nn.Module):
    """
    Atención global aditiva de Bahdanau.

    Para cada estado del decodificador, compara ese estado contra todos los
    estados producidos por el codificador y devuelve un vector de contexto.
    """
    def __init__(self, encoder_hidden_size, decoder_hidden_size, attention_size):
        super().__init__()
        self.energy = nn.Linear(encoder_hidden_size + decoder_hidden_size, attention_size)
        self.score = nn.Linear(attention_size, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, encoder_mask=None):
        """
        decoder_hidden: [batch, decoder_hidden_size]
        encoder_outputs: [batch, src_len, encoder_hidden_size]
        encoder_mask: [batch, src_len]
        """
        batch_size, src_len, _ = encoder_outputs.shape

        decoder_repetido = decoder_hidden.unsqueeze(1).expand(batch_size, src_len, -1)
        energia = torch.tanh(self.energy(torch.cat([decoder_repetido, encoder_outputs], dim=-1)))
        puntajes = self.score(energia).squeeze(-1)

        pesos = masked_softmax(puntajes, encoder_mask, dim=-1)
        contexto = torch.bmm(pesos.unsqueeze(1), encoder_outputs).squeeze(1)

        return contexto, pesos

class Seq2SeqConAtencionGlobal(nn.Module):
    """
    Modelo Seq2Seq mínimo con atención global.

    La corrección principal respecto a una versión demasiado simplificada es que
    la atención se calcula en cada paso del decodificador, no solo una vez al final.
    """
    def __init__(
        self,
        input_dim,
        output_dim,
        embedding_dim,
        encoder_hidden_size,
        decoder_hidden_size,
        attention_size,
        padding_idx=0,
    ):
        super().__init__()
        self.padding_idx = padding_idx

        self.encoder_embedding = nn.Embedding(input_dim, embedding_dim, padding_idx=padding_idx)
        self.decoder_embedding = nn.Embedding(output_dim, embedding_dim, padding_idx=padding_idx)

        self.encoder = nn.GRU(
            embedding_dim,
            encoder_hidden_size,
            batch_first=True,
        )
        self.decoder = nn.GRU(
            embedding_dim + encoder_hidden_size,
            decoder_hidden_size,
            batch_first=True,
        )

        self.inicializar_decoder = nn.Linear(encoder_hidden_size, decoder_hidden_size)
        self.atencion = AtencionBahdanau(encoder_hidden_size, decoder_hidden_size, attention_size)
        self.salida = nn.Linear(decoder_hidden_size + encoder_hidden_size, output_dim)

    def forward(self, src, trg, src_lengths):
        """
        src: [batch, src_len]
        trg: [batch, trg_len]
        src_lengths: [batch]
        """
        src_len = src.size(1)
        src_mask = crear_mascara_por_longitudes(src_lengths, src_len)

        src_emb = self.encoder_embedding(src)

        # Empaquetar evita que la GRU del encoder aprenda de posiciones de padding.
        packed = nn.utils.rnn.pack_padded_sequence(
            src_emb,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, hidden = self.encoder(packed)
        encoder_outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=src_len,
        )

        hidden_decoder = torch.tanh(self.inicializar_decoder(hidden[-1])).unsqueeze(0)

        logits_por_paso = []
        atenciones_por_paso = []

        for t in range(trg.size(1)):
            token_t = trg[:, t].unsqueeze(1)
            emb_t = self.decoder_embedding(token_t)

            contexto, pesos_atencion = self.atencion(
                hidden_decoder[-1],
                encoder_outputs,
                src_mask,
            )

            entrada_decoder = torch.cat([emb_t, contexto.unsqueeze(1)], dim=-1)
            salida_decoder, hidden_decoder = self.decoder(entrada_decoder, hidden_decoder)

            logits_t = self.salida(torch.cat([salida_decoder.squeeze(1), contexto], dim=-1))
            logits_por_paso.append(logits_t.unsqueeze(1))
            atenciones_por_paso.append(pesos_atencion.unsqueeze(1))

        logits = torch.cat(logits_por_paso, dim=1)
        pesos_atencion = torch.cat(atenciones_por_paso, dim=1)

        return logits, pesos_atencion

# Demostración con datos sintéticos.
batch_size = 2
src_len = 6
trg_len = 4
input_dim = 40
output_dim = 50

src = torch.tensor([
    [5, 7, 9, 11, 13, 0],  # la última posición es padding
    [4, 6, 8, 10, 12, 14],
], dtype=torch.long)
src_lengths = torch.tensor([5, 6], dtype=torch.long)
trg = torch.tensor([
    [2, 3, 4, 5],
    [2, 8, 9, 10],
], dtype=torch.long)

modelo_global = Seq2SeqConAtencionGlobal(
    input_dim=input_dim,
    output_dim=output_dim,
    embedding_dim=16,
    encoder_hidden_size=32,
    decoder_hidden_size=32,
    attention_size=24,
    padding_idx=0,
)

logits, pesos = modelo_global(src, trg, src_lengths)

print("Forma de logits:", logits.shape)
print("Forma de pesos de atención:", pesos.shape)
print("Suma de atención por paso:", pesos.sum(dim=-1))
print("Atención asignada al padding de la primera secuencia:", pesos[0, :, -1])

assert logits.shape == (batch_size, trg_len, output_dim)
assert pesos.shape == (batch_size, trg_len, src_len)
assert torch.allclose(pesos.sum(dim=-1), torch.ones(batch_size, trg_len), atol=1e-5)
assert torch.allclose(pesos[0, :, -1], torch.zeros(trg_len), atol=1e-6)


#### **Mecanismo de atención local**

**Descripción general**

El mecanismo de atención local, propuesto por Luong (2015), reduce la complejidad computacional al limitar el alcance de la atención a una ventana local alrededor de cada posición de la secuencia de entrada. Este enfoque es particularmente útil en secuencias largas, donde la atención global puede ser computacionalmente costosa.

**Ecuaciones y cálculos**

El mecanismo de atención local se define a través de los siguientes pasos:

1. **Predicción de la posición de atención**:
   Primero, se predice una posición de atención $p_j$ para cada paso de tiempo $j$ del decodificador. Esto puede hacerse mediante una simple función lineal o una red neuronal:
   

   $$
   p_j = S \cdot \sigma(W_p s_{j-1})
   $$ 

   donde $S$ es la longitud de la secuencia de entrada, $\sigma$ es la función sigmoide, $W_p$ es una matriz de peso aprendible, y $s_{j-1}$ es el estado oculto del decodificador en el paso $j-1$.


2. **Definición de la ventana local**:
   Se define una ventana local de tamaño $2D + 1$ centrada en $p_j$. Los límites de la ventana se calculan como:

   $$
   [p_j - D, p_j + D]
  $$ 


3. **Cálculo de puntajes de atención dentro de la ventana**:
   Los puntajes de atención $e_{ij}$ se calculan solo para las posiciones dentro de la ventana local:


   $$
   e_{ij} = v^T \tanh(W_1 h_i + W_2 s_{j-1})
  $$ 


4. **Normalización de puntajes de atención**:
   Los puntajes de atención se normalizan usando la función softmax para obtener los pesos de atención $\alpha_{ij}$:

   $$
   \alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in [p_j - D, p_j + D]} \exp(e_{ik})}
  $$ 

5. **Cálculo del vector de contexto**:
   El vector de contexto $c_j$ se calcula como una combinación ponderada de los estados ocultos del codificador dentro de la ventana local:

   $$
   c_j = \sum_{i \in [p_j - D, p_j + D]} \alpha_{ij} h_i
  $$ 

<img src="https://lilianweng.github.io/posts/2018-06-24-attention/luong2015-fig2-3.png" width="600">


La siguiente implementación en PyTorch muestra cómo se puede construir un mecanismo de atención local dentro de un modelo seq2seq:

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class AtencionLocalLuong(nn.Module):
    """
    Atención local inspirada en Luong.

    La diferencia clave respecto a la atención global es que se restringe la
    distribución de atención a una ventana alrededor de una posición central.
    """
    def __init__(self, encoder_hidden_size, decoder_hidden_size, window_size=2):
        super().__init__()
        self.window_size = window_size
        self.proyectar_encoder = nn.Linear(encoder_hidden_size, decoder_hidden_size, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, centros, encoder_mask=None):
        """
        decoder_hidden: [batch, decoder_hidden_size]
        encoder_outputs: [batch, src_len, encoder_hidden_size]
        centros: [batch], posición central de la ventana para cada ejemplo
        encoder_mask: [batch, src_len]
        """
        batch_size, src_len, _ = encoder_outputs.shape

        claves = self.proyectar_encoder(encoder_outputs)
        puntajes = torch.bmm(claves, decoder_hidden.unsqueeze(-1)).squeeze(-1)

        posiciones = torch.arange(src_len, device=encoder_outputs.device).unsqueeze(0)
        mascara_local = (posiciones - centros.unsqueeze(1)).abs() <= self.window_size

        if encoder_mask is not None:
            mascara_local = mascara_local & encoder_mask.to(torch.bool)

        pesos = masked_softmax(puntajes, mascara_local, dim=-1)
        contexto = torch.bmm(pesos.unsqueeze(1), encoder_outputs).squeeze(1)

        return contexto, pesos, mascara_local

# Demostración con datos sintéticos.
batch_size = 2
src_len = 8
encoder_hidden_size = 16
decoder_hidden_size = 16

encoder_outputs = torch.randn(batch_size, src_len, encoder_hidden_size)
decoder_hidden = torch.randn(batch_size, decoder_hidden_size)

src_lengths = torch.tensor([7, 5])
encoder_mask = crear_mascara_por_longitudes(src_lengths, src_len)

# Cada ejemplo atiende alrededor de una zona local diferente.
centros = torch.tensor([3, 2])

atencion_local = AtencionLocalLuong(
    encoder_hidden_size=encoder_hidden_size,
    decoder_hidden_size=decoder_hidden_size,
    window_size=1,
)

contexto_local, pesos_locales, mascara_local = atencion_local(
    decoder_hidden,
    encoder_outputs,
    centros,
    encoder_mask,
)

print("Forma del contexto local:", contexto_local.shape)
print("Forma de los pesos locales:", pesos_locales.shape)
print("Máscara local usada:")
print(mascara_local)
print("Pesos de atención local:")
print(pesos_locales)
print("Suma de pesos por ejemplo:", pesos_locales.sum(dim=-1))

assert contexto_local.shape == (batch_size, encoder_hidden_size)
assert pesos_locales.shape == (batch_size, src_len)
assert torch.allclose(pesos_locales.sum(dim=-1), torch.ones(batch_size), atol=1e-5)
assert torch.all(pesos_locales[~mascara_local] == 0)


#### **Mecanismo de atención jerárquica**

**Descripción general**

La atención jerárquica se utiliza para manejar estructuras de datos complejas y de múltiples niveles, como documentos largos divididos en párrafos, párrafos divididos en oraciones y oraciones divididas en palabras. Este mecanismo aplica la atención en dos niveles: a nivel de palabra dentro de cada oración y a nivel de oración dentro del documento. Esta estructura permite capturar dependencias tanto locales como globales de manera eficiente.

**Ecuaciones y cálculos**

El proceso de atención jerárquica se puede dividir en dos fases principales:

1. **Atención a nivel de palabra**:
   Primero, se aplica la atención para cada palabra dentro de cada oración. Supongamos que una oración $ \text{sentence}_i $ contiene $ T_i $ palabras y sus representaciones de palabra son $ \{h_{i1}, h_{i2}, \ldots, h_{iT_i}\} $. La atención a nivel de palabra se calcula de la siguiente manera:

   $$
   e_{ij} = v_1^T \tanh(W_1 h_{ij} + b_1)
   $$

   donde $ W_1 $ y $ v_1 $ son parámetros aprendibles, y $ b_1 $ es un vector de sesgo.

   Los pesos de atención se obtienen aplicando la función softmax a los puntajes $ e_{ij} $:

   $$
   \alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_i} \exp(e_{ik})}
   $$

   El vector de contexto para la oración $ i $ se calcula como una combinación ponderada de las representaciones de palabra:

   $$
   c_i = \sum_{j=1}^{T_i} \alpha_{ij} h_{ij}
   $$

2. **Atención a nivel de oración**:
   Una vez obtenidos los vectores de contexto $ \{c_1, c_2, \ldots, c_N\} $ para todas las oraciones en un documento (donde $ N $ es el número de oraciones en el documento), se aplica la atención a nivel de oración:

   $$
   e_i = v_2^T \tanh(W_2 c_i + b_2)
   $$

   Los pesos de atención a nivel de oración se obtienen aplicando la función softmax a los puntajes $ e_i $:

   $$
   \beta_i = \frac{\exp(e_i)}{\sum_{k=1}^{N} \exp(e_k)}
   $$

   El vector de contexto para el documento se calcula como una combinación ponderada de los vectores de contexto de las oraciones:

   $$
   d = \sum_{i=1}^{N} \beta_i c_i
   $$


La siguiente implementación en PyTorch muestra cómo se puede construir un mecanismo de atención jerárquica:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PoolingAtencionalAditivo(nn.Module):
    """
    Reduce una secuencia a un vector usando atención aditiva.

    A diferencia de una capa lineal que devuelve un vector por token, aquí se
    produce un puntaje escalar por posición para construir una distribución.
    """
    def __init__(self, hidden_size, attention_size):
        super().__init__()
        self.proyeccion = nn.Linear(hidden_size, attention_size)
        self.puntaje = nn.Linear(attention_size, 1, bias=False)

    def forward(self, secuencia, mascara=None):
        """
        secuencia: [batch, seq_len, hidden_size]
        mascara: [batch, seq_len]
        """
        energia = torch.tanh(self.proyeccion(secuencia))
        puntajes = self.puntaje(energia).squeeze(-1)
        pesos = masked_softmax(puntajes, mascara, dim=-1)
        vector = torch.bmm(pesos.unsqueeze(1), secuencia).squeeze(1)
        return vector, pesos

class AtencionJerarquica(nn.Module):
    """
    Red jerárquica con atención a nivel de palabras y a nivel de oraciones.

    Entrada esperada:
    documentos: [batch, num_oraciones, num_palabras, embedding_dim]
    """
    def __init__(
        self,
        embedding_dim,
        word_hidden_size,
        sentence_hidden_size,
        attention_size,
        num_classes,
    ):
        super().__init__()

        assert word_hidden_size % 2 == 0, "word_hidden_size debe ser par para usar GRU bidireccional"
        assert sentence_hidden_size % 2 == 0, "sentence_hidden_size debe ser par para usar GRU bidireccional"

        self.word_encoder = nn.GRU(
            embedding_dim,
            word_hidden_size // 2,
            batch_first=True,
            bidirectional=True,
        )
        self.word_attention = PoolingAtencionalAditivo(word_hidden_size, attention_size)

        self.sentence_encoder = nn.GRU(
            word_hidden_size,
            sentence_hidden_size // 2,
            batch_first=True,
            bidirectional=True,
        )
        self.sentence_attention = PoolingAtencionalAditivo(sentence_hidden_size, attention_size)

        self.classifier = nn.Linear(sentence_hidden_size, num_classes)

    def forward(self, documentos, word_lengths, sentence_lengths):
        """
        documentos: [batch, num_oraciones, num_palabras, embedding_dim]
        word_lengths: [batch, num_oraciones]
        sentence_lengths: [batch]
        """
        batch_size, num_oraciones, num_palabras, embedding_dim = documentos.shape

        # Nivel de palabras: procesamos todas las oraciones como un lote plano.
        oraciones = documentos.reshape(batch_size * num_oraciones, num_palabras, embedding_dim)
        longitudes_palabras = word_lengths.reshape(batch_size * num_oraciones).clamp_min(1)

        salidas_palabra, _ = self.word_encoder(oraciones)
        mascara_palabras = crear_mascara_por_longitudes(longitudes_palabras, num_palabras)

        vectores_oracion, pesos_palabra = self.word_attention(
            salidas_palabra,
            mascara_palabras,
        )

        vectores_oracion = vectores_oracion.reshape(batch_size, num_oraciones, -1)
        pesos_palabra = pesos_palabra.reshape(batch_size, num_oraciones, num_palabras)

        # Nivel de oraciones.
        salidas_oracion, _ = self.sentence_encoder(vectores_oracion)
        mascara_oraciones = crear_mascara_por_longitudes(sentence_lengths, num_oraciones)

        vector_documento, pesos_oracion = self.sentence_attention(
            salidas_oracion,
            mascara_oraciones,
        )

        logits = self.classifier(vector_documento)

        return logits, pesos_palabra, pesos_oracion

# Demostración con documentos sintéticos.
batch_size = 2
num_oraciones = 3
num_palabras = 5
embedding_dim = 12
num_classes = 4

documentos = torch.randn(batch_size, num_oraciones, num_palabras, embedding_dim)

# Longitudes reales: la primera muestra tiene 3 oraciones; la segunda, 2.
sentence_lengths = torch.tensor([3, 2])
word_lengths = torch.tensor([
    [5, 4, 3],
    [5, 2, 1],  # la tercera oración de la segunda muestra queda enmascarada por sentence_lengths
])

modelo_jerarquico = AtencionJerarquica(
    embedding_dim=embedding_dim,
    word_hidden_size=20,
    sentence_hidden_size=16,
    attention_size=10,
    num_classes=num_classes,
)

logits_doc, pesos_palabra, pesos_oracion = modelo_jerarquico(
    documentos,
    word_lengths,
    sentence_lengths,
)

print("Forma de logits del documento:", logits_doc.shape)
print("Forma de atención por palabra:", pesos_palabra.shape)
print("Forma de atención por oración:", pesos_oracion.shape)
print("Suma de pesos por oración válida:")
print(pesos_palabra.sum(dim=-1))
print("Suma de pesos por documento:")
print(pesos_oracion.sum(dim=-1))

assert logits_doc.shape == (batch_size, num_classes)
assert pesos_palabra.shape == (batch_size, num_oraciones, num_palabras)
assert pesos_oracion.shape == (batch_size, num_oraciones)
assert torch.allclose(pesos_oracion.sum(dim=-1), torch.ones(batch_size), atol=1e-5)

mascara_oraciones = crear_mascara_por_longitudes(sentence_lengths, num_oraciones)
assert torch.all(pesos_oracion[~mascara_oraciones] == 0)


#### **Mecanismo de atención basada en consultas**

**Descripción general**

La atención basada en consultas, utilizada en modelos como el Transformer, utiliza tres componentes principales: consultas (queries), claves (keys) y valores (values). Este mecanismo permite calcular la atención como una función de similitud entre las consultas y las claves, aplicándola a los valores para obtener una representación ponderada. Este enfoque es altamente eficiente y escalable.

**Ecuaciones y cálculos**

El mecanismo de atención basada en consultas se define a través de los siguientes pasos:

1. **Cálculo de consultas, claves y valores**:
   Las consultas $ Q$, las claves $ K$ y los valores $ V$ se obtienen mediante proyecciones lineales de la entrada:

   $$
   Q = X W_Q, \quad K = X W_K, \quad V = X W_V
   $$

   donde $ W_Q$, $ W_K$ y $ W_V$ son matrices de peso aprendibles.

2. **Cálculo de puntajes de atención**:
   Los puntajes de atención se calculan como el producto punto escalado entre las consultas y las claves:

   $$
   e_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}}
   $$

   donde $ d_k$ es la dimensión de las claves.

3. **Normalización de puntajes de atención**:
   Los puntajes de atención se normalizan usando la función softmax para obtener los pesos de atención $ \alpha_{ij}$:

   $$
   \alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k} \exp(e_{ik})}
   $$

4. **Cálculo del vector de contexto**:
   El vector de contexto $ c_i$ se calcula como una combinación ponderada de los valores:

   $$
   c_i = \sum_{j} \alpha_{ij} V_j
   $$


La siguiente implementación en PyTorch muestra cómo se puede construir un mecanismo de atención basada en consultas

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class AtencionProductoPuntoEscalada(nn.Module):
    """
    Atención basada en consultas, claves y valores.

    Implementa:
        softmax(QK^T / sqrt(d_k)) V
    """
    def forward(self, query, key, value, key_padding_mask=None):
        """
        query: [batch, query_len, d_k]
        key: [batch, key_len, d_k]
        value: [batch, key_len, d_v]
        key_padding_mask: [batch, key_len]
        """
        d_k = query.size(-1)
        puntajes = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

        if key_padding_mask is not None:
            mascara = key_padding_mask.unsqueeze(1).expand(-1, query.size(1), -1)
        else:
            mascara = None

        pesos = masked_softmax(puntajes, mascara, dim=-1)
        contexto = torch.matmul(pesos, value)

        return contexto, pesos

class AtencionBasadaEnConsultas(nn.Module):
    """
    Proyecta una secuencia de consultas, claves y valores antes de aplicar
    producto punto escalado.
    """
    def __init__(self, input_size, hidden_size, value_size=None):
        super().__init__()
        if value_size is None:
            value_size = hidden_size

        self.query = nn.Linear(input_size, hidden_size)
        self.key = nn.Linear(input_size, hidden_size)
        self.value = nn.Linear(input_size, value_size)
        self.atencion = AtencionProductoPuntoEscalada()

    def forward(self, consultas, memoria, memory_mask=None):
        """
        consultas: [batch, query_len, input_size]
        memoria: [batch, memory_len, input_size]
        memory_mask: [batch, memory_len]
        """
        Q = self.query(consultas)
        K = self.key(memoria)
        V = self.value(memoria)

        contexto, pesos = self.atencion(Q, K, V, memory_mask)
        return contexto, pesos

# Demostración con datos sintéticos.
batch_size = 2
query_len = 4
memory_len = 6
input_size = 10
hidden_size = 8
value_size = 12

consultas = torch.randn(batch_size, query_len, input_size)
memoria = torch.randn(batch_size, memory_len, input_size)
memory_lengths = torch.tensor([6, 4])
memory_mask = crear_mascara_por_longitudes(memory_lengths, memory_len)

atencion_qkv = AtencionBasadaEnConsultas(
    input_size=input_size,
    hidden_size=hidden_size,
    value_size=value_size,
)

contexto_qkv, pesos_qkv = atencion_qkv(consultas, memoria, memory_mask)

print("Forma del contexto Q/K/V:", contexto_qkv.shape)
print("Forma de los pesos Q/K/V:", pesos_qkv.shape)
print("Suma de atención por consulta:")
print(pesos_qkv.sum(dim=-1))
print("Pesos sobre padding en el segundo ejemplo:")
print(pesos_qkv[1, :, 4:])

assert contexto_qkv.shape == (batch_size, query_len, value_size)
assert pesos_qkv.shape == (batch_size, query_len, memory_len)
assert torch.allclose(pesos_qkv.sum(dim=-1), torch.ones(batch_size, query_len), atol=1e-5)
assert torch.allclose(pesos_qkv[1, :, 4:], torch.zeros(query_len, 2), atol=1e-6)


#### **Mecanismo de auto-atención (Self-Attention)**

**Descripción general**

El mecanismo de auto-atención, o self-attention, permite que cada elemento de la secuencia preste atención a todos los demás elementos de la misma secuencia. Esto es fundamental para capturar las dependencias a largo plazo en las secuencias y es un componente clave en los modelos Transformer.

**Ecuaciones y cálculos**

El proceso de auto-atención se puede describir mediante los siguientes pasos:

1. **Proyección lineal**:
   Al igual que en la atención multi-cabecera, se proyectan las consultas $Q$, las claves $K$ y los valores $V$:

   $$
   Q = X W_Q, \quad K = X W_K, \quad V = X W_V
   $$

   donde $W_Q$, $W_K$ y $W_V$ son matrices de peso aprendibles.

2. **Cálculo de puntajes de atención**:
   Los puntajes de atención se calculan utilizando el producto punto escalado:

   $$
   e_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}}
   $$

3. **Normalización de puntajes de atención**:
   Los puntajes de atención se normalizan usando la función softmax para obtener los pesos de atención  $\alpha_{ij}$:

   $$
   \alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k} \exp(e_{ik})}
   $$

4. **Cálculo del vector de contexto**:
   El vector de contexto $c_i$ se calcula como una combinación ponderada de los valores:

   $$
   c_i = \sum_{j} \alpha_{ij} V_j
   $$


La siguiente implementación en PyTorch muestra cómo se puede construir un mecanismo de auto-atención:

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class AutoAtencion(nn.Module):
    """
    Self-attention básica.

    La misma secuencia produce Q, K y V. Cada posición puede atender a las demás
    posiciones de la misma secuencia.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.query = nn.Linear(input_size, hidden_size)
        self.key = nn.Linear(input_size, hidden_size)
        self.value = nn.Linear(input_size, hidden_size)
        self.atencion = AtencionProductoPuntoEscalada()

    def forward(self, x, padding_mask=None):
        """
        x: [batch, seq_len, input_size]
        padding_mask: [batch, seq_len]
        """
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        contexto, pesos = self.atencion(Q, K, V, padding_mask)
        return contexto, pesos

# Demostración con datos sintéticos.
batch_size = 2
seq_len = 5
input_size = 10
hidden_size = 16

x = torch.randn(batch_size, seq_len, input_size)
longitudes = torch.tensor([5, 3])
padding_mask = crear_mascara_por_longitudes(longitudes, seq_len)

auto_atencion = AutoAtencion(input_size=input_size, hidden_size=hidden_size)
contexto_self, pesos_self = auto_atencion(x, padding_mask)

print("Forma del contexto de auto-atención:", contexto_self.shape)
print("Forma de los pesos de auto-atención:", pesos_self.shape)
print("Suma de atención por token:")
print(pesos_self.sum(dim=-1))
print("Atención hacia padding en el segundo ejemplo:")
print(pesos_self[1, :, 3:])

assert contexto_self.shape == (batch_size, seq_len, hidden_size)
assert pesos_self.shape == (batch_size, seq_len, seq_len)
assert torch.allclose(pesos_self.sum(dim=-1), torch.ones(batch_size, seq_len), atol=1e-5)
assert torch.allclose(pesos_self[1, :, 3:], torch.zeros(seq_len, 2), atol=1e-6)


#### **Mecanismo de atención multi-cabecera (Multi-Head Attention)**


<img src="https://lilianweng.github.io/posts/2018-06-24-attention/transformer.png" width="600">

**Descripción general**

El mecanismo de atención multi-cabecera, introducido por Vaswani en el modelo Transformer, extiende la idea de la auto-atención al permitir que el modelo se concentre en diferentes partes de la secuencia de entrada de manera simultánea y desde múltiples perspectivas. Esto se logra al tener múltiples "cabeceras" de atención, cada una de las cuales realiza una operación de atención independiente.

**Ecuaciones y cálculos**

El proceso de atención multi-cabecera se puede dividir en varios pasos:

1. **Proyección lineal**:
   Se proyectan las consultas  $Q$, las claves $K$ y los valores $V$ en subespacios diferentes para cada cabecera de atención. Supongamos que tenemos $h$ cabecera de atención y una dimensión de modelo $d_{\text{model}}$. La proyección se realiza de la siguiente manera:

   $$
   Q_h = X W_Q^h, \quad K_h = X W_K^h, \quad V_h = X W_V^h
   $$

   donde $W_Q^h$, $W_K^h$ y $W_V^h$ son matrices de peso específicas para la cabecera  $h$.

2. **Cálculo de puntajes de atención**:
   Para cada cabecera de atención, se calcula el puntaje de atención utilizando el producto punto escalado:

   $$
   e_{ij}^h = \frac{Q_i^h (K_j^h)^T}{\sqrt{d_k}}
   $$

   donde $d_k$ es la dimensión de las claves.

3. **Normalización de puntajes de atención**:

   Los puntajes de atención se normalizan usando la función softmax para obtener los pesos de atención $\alpha_{ij}^h$:

   $$
   \alpha_{ij}^h = \frac{\exp(e_{ij}^h)}{\sum_{k} \exp(e_{ik}^h)}
   $$

4. **Cálculo del vector de contexto**:
   El vector de contexto para cada cabecera de atención se calcula como una combinación ponderada de los valores:

   $$
   c_i^h = \sum_{j} \alpha_{ij}^h V_j^h
   $$

5. **Concatenación y proyección final**:
   Los vectores de contexto de todas las cabeceras se concatenan y se proyectan de nuevo en el espacio original:

   $$
   \text{MultiHead}(Q, K, V) = \text{Concat}(c_i^1, c_i^2, \ldots, c_i^h) W_O
   $$

   donde  $W_O$ es la matriz de peso de proyección final.

La siguiente implementación en PyTorch muestra cómo se puede construir un mecanismo de atención multi-cabecera:

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class AtencionMulticabecera(nn.Module):
    """
    Atención multi-cabecera básica.

    Divide el espacio oculto en varias cabeceras, aplica atención por cabecera y
    luego concatena los resultados.
    """
    def __init__(self, input_size, hidden_size, num_heads):
        super().__init__()

        assert hidden_size % num_heads == 0, "hidden_size debe ser divisible por num_heads"

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.query = nn.Linear(input_size, hidden_size)
        self.key = nn.Linear(input_size, hidden_size)
        self.value = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, hidden_size)

    def dividir_cabeceras(self, tensor):
        """
        [batch, seq_len, hidden_size] -> [batch, heads, seq_len, head_dim]
        """
        batch_size, seq_len, _ = tensor.shape
        tensor = tensor.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return tensor.transpose(1, 2)

    def unir_cabeceras(self, tensor):
        """
        [batch, heads, seq_len, head_dim] -> [batch, seq_len, hidden_size]
        """
        batch_size, num_heads, seq_len, head_dim = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.view(batch_size, seq_len, num_heads * head_dim)

    def forward(self, x, padding_mask=None, attention_mask=None):
        """
        x: [batch, seq_len, input_size]
        padding_mask: [batch, seq_len]
        attention_mask: [seq_len, seq_len] o [batch, seq_len, seq_len]
        """
        Q = self.dividir_cabeceras(self.query(x))
        K = self.dividir_cabeceras(self.key(x))
        V = self.dividir_cabeceras(self.value(x))

        puntajes = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        mascara = None

        if padding_mask is not None:
            mascara_padding = padding_mask[:, None, None, :].to(torch.bool)
            mascara = mascara_padding.expand(-1, self.num_heads, x.size(1), -1)

        if attention_mask is not None:
            if attention_mask.dim() == 2:
                mascara_atencion = attention_mask[None, None, :, :].to(torch.bool)
            elif attention_mask.dim() == 3:
                mascara_atencion = attention_mask[:, None, :, :].to(torch.bool)
            else:
                raise ValueError("attention_mask debe tener 2 o 3 dimensiones")

            if mascara is None:
                mascara = mascara_atencion
            else:
                mascara = mascara & mascara_atencion

        pesos = masked_softmax(puntajes, mascara, dim=-1)
        contexto = torch.matmul(pesos, V)

        contexto = self.unir_cabeceras(contexto)
        salida = self.output(contexto)

        return salida, pesos

# Demostración con datos sintéticos.
batch_size = 2
seq_len = 6
input_size = 12
hidden_size = 24
num_heads = 4

x = torch.randn(batch_size, seq_len, input_size)
longitudes = torch.tensor([6, 4])
padding_mask = crear_mascara_por_longitudes(longitudes, seq_len)

multi_head = AtencionMulticabecera(
    input_size=input_size,
    hidden_size=hidden_size,
    num_heads=num_heads,
)

salida_mha, pesos_mha = multi_head(x, padding_mask=padding_mask)

print("Forma de la salida multi-cabecera:", salida_mha.shape)
print("Forma de los pesos multi-cabecera:", pesos_mha.shape)
print("Suma de pesos por cabecera y token:")
print(pesos_mha.sum(dim=-1))
print("Atención hacia padding en el segundo ejemplo:")
print(pesos_mha[1, :, :, 4:])

assert salida_mha.shape == (batch_size, seq_len, hidden_size)
assert pesos_mha.shape == (batch_size, num_heads, seq_len, seq_len)
assert torch.allclose(
    pesos_mha.sum(dim=-1),
    torch.ones(batch_size, num_heads, seq_len),
    atol=1e-5,
)
assert torch.allclose(
    pesos_mha[1, :, :, 4:],
    torch.zeros(num_heads, seq_len, 2),
    atol=1e-6,
)


Estos mecanismos son fundamentales para el funcionamiento de los modelos Transformer, permitiendo capturar dependencias a largo plazo y manejar grandes cantidades de datos de manera eficiente. 